# AQRS Google Colab Setup
This notebook mounts Google Drive, clones the repo into Drive, installs dependencies, sets `PYTHONPATH`, and verifies parquet access.
Do NOT run full benchmarks from this notebook until you have copied `data/raw` files into Drive.

In [50]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
import pandas as pd, os
src="/content/repo/data/raw/EURUSD/1m.parquet"
dst="/content/repo/data/raw/EURUSD/15m.parquet"
os.makedirs(os.path.dirname(dst), exist_ok=True)
df = pd.read_parquet(src)
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
else:
    df.index = pd.to_datetime(df.index)
agg = {}
for c in ['open','high','low','close','volume']:
    if c in df.columns:
        if c=='open': agg[c]='first'
        elif c=='high': agg[c]='max'
        elif c=='low': agg[c]='min'
        elif c=='close': agg[c]='last'
        elif c=='volume': agg[c]='sum'
res = df.resample('15min').agg(agg).dropna()
res.reset_index(inplace=True)
res.to_parquet(dst)
print('Wrote', dst, 'rows=', len(res))

Wrote /content/repo/data/raw/EURUSD/15m.parquet rows= 398113


In [ ]:
%%bash
cd /content/repo
export PYTHONPATH=$PWD:$PYTHONPATH
mkdir -p /content/drive/MyDrive/autonomous-quant-research-system/artifacts
TIMEFRAMES=15m MODELS=baseline python3 -u scripts/eurusd_benchmark_wfv.py --smoke |& tee /content/drive/MyDrive/autonomous-quant-research-system/artifacts/eurusd_benchmark_wfv_colab.log || true
tail -n 200 /content/drive/MyDrive/autonomous-quant-research-system/artifacts/eurusd_benchmark_wfv_colab.log

In [ ]:
%%bash
cd /content/repo
export PYTHONPATH=/content/repo:$PYTHONPATH
echo "PYTHONPATH=$PYTHONPATH"

python3 - <<'PY'
import importlib.util, traceback
spec = importlib.util.find_spec("app.models.lightgbm_classifier")
print("find_spec:", spec)
if spec:
    print("module file:", spec.origin)
try:
    import app.models.lightgbm_classifier as mod
    print("Imported OK:", getattr(mod, "__file__", mod))
except Exception:
    traceback.print_exc()
PY

# If import above succeeded, run the smoke test (log saved to Drive)
TIMEFRAMES=15m MODELS=baseline python3 -u scripts/eurusd_benchmark_wfv.py --smoke |& tee /content/drive/MyDrive/autonomous-quant-research-system/artifacts/eurusd_benchmark_wfv_colab.log || true
tail -n 200 /content/drive/MyDrive/autonomous-quant-research-system/artifacts/eurusd_benchmark_wfv_colab.log || true

In [ ]:
# Create missing LightGBM model file in the Colab repo (writes from notebook)
import os
content = '''from lightgbm import LGBMClassifier


class LightGBMClassifierModel:
    def __init__(self, random_state: int = 42):
        self.random_state = random_state

    def train(
        self,
        X,
        y,
    ):

        model = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=self.random_state,
            verbosity=-1,
        )

        model.fit(
            X,
            y,
        )

        return model
'''
path = "/content/repo/app/models/lightgbm_classifier.py"
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    f.write(content)
print("Wrote", path)